# Evaluasi Kuantitatif Chatbot (LLM-as-a-judge)
Notebook ini mengambil data dari Supabase (history chat) dan Aiven MySQL (progress user materi), lalu menggunakan beberapa model LLM via OpenRouter sebagai juri penilai.

In [ ]:
import os
import json
import time
import itertools
from urllib.parse import urlparse

import pymysql
import pandas as pd
import requests
from dotenv import load_dotenv
from supabase import create_client, Client
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score

load_dotenv()

SUPABASE_URL = os.environ.get('SUPABASE_URL')
SUPABASE_KEY = os.environ.get('SUPABASE_SERVICE_ROLE_KEY')
BACKEND_LLM_API_BASE = os.environ.get('BACKEND_LLM_API_BASE', 'https://backend-65ah.vercel.app/api/llm/openrouter').rstrip('/')
DATABASE_URL = os.environ.get('DATABASE_URL')
TARGET_USER_ID = os.environ.get('TARGET_USER_ID')

missing_envs = []
if not SUPABASE_URL:
    missing_envs.append('SUPABASE_URL')
if not SUPABASE_KEY:
    missing_envs.append('SUPABASE_SERVICE_ROLE_KEY')
if not DATABASE_URL:
    missing_envs.append('DATABASE_URL')

if missing_envs:
    print('Missing environment variables:', ', '.join(missing_envs))

if not BACKEND_LLM_API_BASE:
    print('BACKEND_LLM_API_BASE belum diisi. RUN_LLM_EVAL=True akan gagal saat call evaluator model.')

supabase = None
if SUPABASE_URL and SUPABASE_KEY:
    try:
        supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    except Exception as e:
        print('Failed to setup Supabase client:', e)

if TARGET_USER_ID:
    print(f'Filter aktif: hanya user_id={TARGET_USER_ID}')

## 1. Ambil Data Percakapan dari Supabase

In [17]:
def fetch_all_rows(client: Client, table_name: str, page_size: int = 1000):
    rows = []
    start = 0
    while True:
        end = start + page_size - 1
        resp = client.table(table_name).select('*').range(start, end).execute()
        batch = resp.data or []
        rows.extend(batch)
        if len(batch) < page_size:
            break
        start += page_size
    return rows

try:
    if supabase is None:
        raise RuntimeError('Supabase client is not initialized. Check SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY.')

    sessions = fetch_all_rows(supabase, 'chat_sessions')
    messages = fetch_all_rows(supabase, 'chat_messages')

    df_sessions = pd.DataFrame(sessions)
    df_messages = pd.DataFrame(messages)

    print(f'Total sessions: {len(df_sessions)}')
    print(f'Total messages: {len(df_messages)}')
    if not df_messages.empty:
        display(df_messages.head())
except Exception as e:
    print('Supabase fetch error:', e)
    df_messages = pd.DataFrame()
    df_sessions = pd.DataFrame()

Total sessions: 391
Total messages: 1988


,id,session_id,role,content,metadata,token_count,created_at
0,9ef5d0b9-09e3-4b07-9fa5-7818a8071c26,6d389152-5016-4d99-bb10-f76d582bc41e,user,Bisa kasih tips belajar cepat untuk bagian ini?,"{'mock': True, 'turn': 1, 'sim_day': 1}",None,2026-03-05T00:51:00+00:00
1,edea6acc-efdf-492d-9a60-fe71f7c4c617,6d389152-5016-4d99-bb10-f76d582bc41e,assistant,"Halo Ralphael, untuk pertanyaanmu: Bisa kasih ...","{'mock': True, 'turn': 1, 'sim_day': 1}",None,2026-03-05T00:51:00+00:00
2,a35797bc-d9d2-4bf2-9585-aa62701353ab,319e918d-71a8-4414-8969-dd53c3e39033,user,Boleh jelaskan ringkas topik ini?,"{'mock': True, 'turn': 1, 'sim_day': 1}",None,2026-03-05T04:45:00+00:00
3,749d00de-0704-4c95-a84c-e49e63ef858d,319e918d-71a8-4414-8969-dd53c3e39033,assistant,"Halo Ralphael, untuk pertanyaanmu: Boleh jelas...","{'mock': True, 'turn': 1, 'sim_day': 1}",None,2026-03-05T04:45:00+00:00
4,465b9145-5ff1-43d2-ae12-c1ce48b6d97b,319e918d-71a8-4414-8969-dd53c3e39033,user,Boleh jelaskan ringkas topik ini?,"{'mock': True, 'turn': 2, 'sim_day': 1}",None,2026-03-05T04:51:00+00:00


## 2. Ambil Data Context User & Materi dari MySQL Server (Levelearn DB)
Kita gunakan urllib untuk mem-parse connection string dari Prisma (DATABASE_URL) agar kompatibel dengan PyMySQL.

In [8]:
df_users = pd.DataFrame()
if DATABASE_URL:
    try:
        parsed_url = urlparse(DATABASE_URL)
        conn = pymysql.connect(
            host=parsed_url.hostname,
            user=parsed_url.username,
            password=parsed_url.password,
            database=parsed_url.path[1:],
            port=parsed_url.port or 3306
        )

        query = 'SELECT id, name, username, points FROM users'
        df_users = pd.read_sql_query(query, conn)
        conn.close()
        print('Berhasil tersambung ke MySQL Aiven Server!')
        print(f'Total User: {len(df_users)}')
    except Exception as e:
        print('Database error:', e)

df = pd.DataFrame()
if not df_messages.empty and not df_sessions.empty:
    required_session_cols = {'id', 'user_id'}
    if not required_session_cols.issubset(df_sessions.columns):
        print('chat_sessions does not contain required columns:', required_session_cols)
    else:
        df = pd.merge(
            df_messages,
            df_sessions[['id', 'user_id']],
            left_on='session_id',
            right_on='id',
            suffixes=('', '_session')
        )

        if TARGET_USER_ID:
            df = df[df['user_id'].astype(str) == str(TARGET_USER_ID)].copy()
            print(f"Total messages setelah filter user_id={TARGET_USER_ID}: {len(df)}")

        if not df_users.empty:
            # Normalisasi tipe key agar join user_id vs users.id tidak gagal karena beda tipe.
            df['user_id_norm'] = df['user_id'].astype(str)
            df_users_merge = df_users.copy()
            df_users_merge['id'] = df_users_merge['id'].astype(str)

            df = pd.merge(
                df,
                df_users_merge,
                left_on='user_id_norm',
                right_on='id',
                how='left'
            )
            df = df.drop(columns=['user_id_norm'])

            if 'name' in df.columns:
                df['name'] = df['name'].fillna('Anonim')
            if 'points' in df.columns:
                df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
        else:
            df['name'] = 'Anonim'
            df['points'] = 0

        sort_cols = [c for c in ['user_id', 'session_id', 'created_at'] if c in df.columns]
        if sort_cols:
            df = df.sort_values(sort_cols).reset_index(drop=True)

        display(df.head())

Berhasil tersambung ke MySQL Aiven Server!
Total User: 56


C:\Users\Asus\AppData\Local\Temp\ipykernel_31584\3401501936.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_users = pd.read_sql_query(query, conn)


,id_x,session_id,role,content,metadata,token_count,created_at,id_session,user_id,id_y,name,username,points
0,839a39fe-349d-4bf2-baf3-44d3bc76a16e,16b45838-7640-414b-a22d-52f3f92a8da4,user,materi apa sekarang yg kupelajari?,{},None,2026-02-22T03:19:41.550914+00:00,16b45838-7640-414b-a22d-52f3f92a8da4,4.0,NaN,Anonim,NaN,0.0
1,e321c604-8e7a-4477-97f7-360a0bc2de0c,16b45838-7640-414b-a22d-52f3f92a8da4,assistant,Halo Benhard! Senang sekali bisa menemani bela...,{},None,2026-02-22T03:19:41.550914+00:00,16b45838-7640-414b-a22d-52f3f92a8da4,4.0,NaN,Anonim,NaN,0.0
2,298eb087-e872-4640-9ff2-dd894472b8de,16b45838-7640-414b-a22d-52f3f92a8da4,user,bisa kah kamu jelaskan gambar yang ada di mate...,{},None,2026-02-22T03:37:42.855817+00:00,16b45838-7640-414b-a22d-52f3f92a8da4,4.0,NaN,Anonim,NaN,0.0
3,8afe2517-e6af-4097-932d-3e642b2de4a1,16b45838-7640-414b-a22d-52f3f92a8da4,assistant,Halo Benhard! Senang sekali kamu bertanya tent...,{},None,2026-02-22T03:37:42.855817+00:00,16b45838-7640-414b-a22d-52f3f92a8da4,4.0,NaN,Anonim,NaN,0.0
4,1c65c508-b929-481b-8a15-f85579ce5772,1e64d54e-1d18-401d-9d74-5fbf29a723e1,user,apa materi yang sedang kita pelajari?,{},None,2026-02-22T08:13:51.311051+00:00,1e64d54e-1d18-401d-9d74-5fbf29a723e1,4.0,NaN,Anonim,NaN,0.0


## 3. Pemrosesan Pasangan (Prompt User - Balasan Chatbot)

In [9]:
pairs = []
if not df.empty:
    sort_cols = [c for c in ['user_id', 'session_id', 'created_at'] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    current_user_msg = None
    turn_counter_by_session = {}

    for _, row in df.iterrows():
        role = row.get('role')
        if role == 'user':
            current_user_msg = row
        elif role == 'assistant' and current_user_msg is not None:
            session_id = row['session_id']
            turn_counter_by_session[session_id] = turn_counter_by_session.get(session_id, 0) + 1

            user_name = row.get('name')
            if pd.isna(user_name) or str(user_name).strip() == '':
                user_name = 'Anonim'

            user_points = row.get('points')
            if pd.isna(user_points):
                user_points = 0

            pairs.append({
                'eval_id': f"{session_id}_{turn_counter_by_session[session_id]}",
                'session_id': session_id,
                'turn_index': turn_counter_by_session[session_id],
                'user_id': row.get('user_id'),
                'user_name': user_name,
                'user_points': user_points,
                'user_message': current_user_msg.get('content', ''),
                'chatbot_reply': row.get('content', ''),
                'created_at': row.get('created_at')
            })
            current_user_msg = None

df_pairs = pd.DataFrame(pairs)
print(f'Total user-assistant interaction pairs: {len(df_pairs)}')
if not df_pairs.empty:
    display(df_pairs.head())

Total user-assistant interaction pairs: 14


,eval_id,session_id,turn_index,user_id,user_name,user_points,user_message,chatbot_reply,created_at
0,16b45838-7640-414b-a22d-52f3f92a8da4_1,16b45838-7640-414b-a22d-52f3f92a8da4,1,4.0,Anonim,0.0,materi apa sekarang yg kupelajari?,Halo Benhard! Senang sekali bisa menemani bela...,2026-02-22T03:19:41.550914+00:00
1,16b45838-7640-414b-a22d-52f3f92a8da4_2,16b45838-7640-414b-a22d-52f3f92a8da4,2,4.0,Anonim,0.0,bisa kah kamu jelaskan gambar yang ada di mate...,Halo Benhard! Senang sekali kamu bertanya tent...,2026-02-22T03:37:42.855817+00:00
2,1e64d54e-1d18-401d-9d74-5fbf29a723e1_1,1e64d54e-1d18-401d-9d74-5fbf29a723e1,1,4.0,Anonim,0.0,apa materi yang sedang kita pelajari?,Halo Benhard! Senang sekali bisa menemani bela...,2026-02-22T08:13:51.311051+00:00
3,76faea2a-e314-451d-9cf6-e5aa73c29896_1,76faea2a-e314-451d-9cf6-e5aa73c29896,1,4.0,Anonim,0.0,bagaimana dengan hasil kuis saya?,"Halo Benhard! Wah, nilai kuis kamu 0 ya? Janga...",2026-02-22T08:38:49.956199+00:00
4,76faea2a-e314-451d-9cf6-e5aa73c29896_2,76faea2a-e314-451d-9cf6-e5aa73c29896,2,4.0,Anonim,0.0,apa yang sedang kita pelajari saat ini?,Halo Benhard! Jangan khawatir soal nilai kuis ...,2026-02-23T09:09:10.12458+00:00


## 4. Evaluasi Menggunakan Multi LLM-as-a-judge (OpenRouter)
Prioritas model: Kimi 2.5, GLM 5, dan Qwen 3.5. Jika tidak tersedia/kuota habis, sistem fallback ke model free tier lain.

In [ ]:
# Konfigurasi evaluasi
RUN_LLM_EVAL = True  # Live call tetap dijaga oleh preflight + hard-stop guardrails
VALIDATION_MODE = 'consistency_only'  # opsi: 'consistency_only' atau 'consistency_human'
SAMPLE_SIZE = 30
STRATIFY_COL = 'user_id'  # Stratified sampling berdasarkan user
SLEEP_BETWEEN_CALLS_SEC = 1.0
MAX_ITEMS_PER_RUN = 30
MAX_TOTAL_JUDGE_CALLS = 90
MAX_CONSECUTIVE_ITEM_FAILURES = 5
MAX_ITEM_FAILURE_RATE = 0.40

OPENROUTER_API_URL = f'{BACKEND_LLM_API_BASE}/chat/completions'
OPENROUTER_MODELS_URL = f'{BACKEND_LLM_API_BASE}/models'

# Cari kandidat target lebih dulu: Kimi 2.5, GLM 5, Qwen 3.5
PRIORITY_MODEL_QUERIES = [
    ('Kimi 2.5', ['moonshotai/kimi-k2.5', 'kimi-k2.5']),
    ('GLM 5', ['z-ai/glm-5', 'glm-5']),
    ('Qwen 3.5', ['qwen/qwen3.5', 'qwen3.5'])
]

# Fallback free-tier jika target di atas tidak punya varian free atau gagal dipakai
FALLBACK_FREE_MODEL_HINTS = [
    'qwen/qwen3-next-80b-a3b-instruct:free',
    'z-ai/glm-4.5-air:free',
    'qwen/qwen3-coder:free',
    'qwen/qwen3-4b:free'
]

MAX_EVALUATOR_MODELS = 3
MAX_RETRIES_PER_MODEL = 2
MIN_VALID_JUDGES = 2
JUDGE_MODE = 'single'  # opsi: 'single' atau 'panel'
SINGLE_JUDGE_MODEL = 'qwen/qwen3-next-80b-a3b-instruct:free'

if JUDGE_MODE == 'single':
    MIN_VALID_JUDGES = 1

# Gunakan mode fixed agar model evaluator konsisten antar-run.
EVALUATOR_SELECTION_MODE = 'fixed'  # opsi: 'fixed' atau 'auto'
FIXED_EVALUATOR_MODELS = [
    'qwen/qwen3-next-80b-a3b-instruct:free',
    'z-ai/glm-4.5-air:free',
    'mistralai/mistral-small-3.1-24b-instruct:free'
]


def _safe_float(v, default=0.0):
    try:
        return float(v)
    except Exception:
        return default


def _is_free_model(model_obj):
    pricing = model_obj.get('pricing', {}) or {}
    prompt_cost = _safe_float(pricing.get('prompt', '0'))
    completion_cost = _safe_float(pricing.get('completion', '0'))
    model_id = (model_obj.get('id') or '').lower()
    return model_id.endswith(':free') or (prompt_cost == 0 and completion_cost == 0)


def fetch_openrouter_model_catalog():
    response = requests.get(OPENROUTER_MODELS_URL, timeout=30)
    response.raise_for_status()
    payload = response.json()
    return payload.get('data', []) if isinstance(payload, dict) else []


def run_backend_route_smoke_test():
    result = {'ok': False, 'models_ok': False, 'chat_ok': False, 'details': []}

    if not BACKEND_LLM_API_BASE:
        result['details'].append('BACKEND_LLM_API_BASE kosong.')
        return result

    try:
        models_resp = requests.get(OPENROUTER_MODELS_URL, timeout=20)
        if models_resp.status_code < 400:
            result['models_ok'] = True
        else:
            result['details'].append(f'Models endpoint status {models_resp.status_code}.')
    except Exception as e:
        result['details'].append(f'Models endpoint error: {e}')

    try:
        # Deliberately send minimal payload to verify route wiring only.
        chat_resp = requests.post(OPENROUTER_API_URL, json={}, timeout=20)
        if chat_resp.status_code < 500 and chat_resp.status_code != 404:
            result['chat_ok'] = True
        else:
            result['details'].append(f'Chat endpoint status {chat_resp.status_code}.')
    except Exception as e:
        result['details'].append(f'Chat endpoint error: {e}')

    result['ok'] = result['models_ok'] and result['chat_ok']
    return result


def search_models(catalog, keyword_list):
    matches = []
    keywords = [k.lower() for k in keyword_list]
    for model in catalog:
        model_id = (model.get('id') or '').lower()
        model_name = (model.get('name') or '').lower()
        if any(k in model_id or k in model_name for k in keywords):
            matches.append(model)
    return matches


def select_evaluator_models(catalog):
    if JUDGE_MODE == 'single':
        available_ids = {m.get('id') for m in catalog if m.get('id')}
        if SINGLE_JUDGE_MODEL in available_ids:
            print(f'Mode evaluator: SINGLE ({SINGLE_JUDGE_MODEL})')
            return [SINGLE_JUDGE_MODEL]

        print(f'Peringatan: SINGLE_JUDGE_MODEL tidak ditemukan di katalog: {SINGLE_JUDGE_MODEL}')
        fallback = next((m.get('id') for m in catalog if _is_free_model(m) and m.get('id')), None)
        if fallback:
            print(f'Menggunakan fallback single judge: {fallback}')
            return [fallback]

        return []

    if EVALUATOR_SELECTION_MODE == 'fixed':
        available_ids = {m.get('id') for m in catalog if m.get('id')}
        selected = [mid for mid in FIXED_EVALUATOR_MODELS if mid in available_ids]
        missing = [mid for mid in FIXED_EVALUATOR_MODELS if mid not in available_ids]

        print('Mode pemilihan evaluator: FIXED')
        if missing:
            print(f'Peringatan: model fixed tidak ditemukan di katalog: {missing}')

        selected = selected[:MAX_EVALUATOR_MODELS]
        print('Model evaluator terpilih:')
        for i, mid in enumerate(selected, start=1):
            print(f'{i}. {mid}')

        return selected

    selected_ids = []

    print('Mencari model prioritas (Kimi 2.5, GLM 5, Qwen 3.5)...')
    for label, queries in PRIORITY_MODEL_QUERIES:
        matches = search_models(catalog, queries)
        matches_sorted = sorted(matches, key=lambda m: m.get('created', 0), reverse=True)

        if matches_sorted:
            free_matches = [m for m in matches_sorted if _is_free_model(m)]
            chosen = free_matches[0] if free_matches else None

            preview_ids = [m.get('id') for m in matches_sorted[:5]]
            print(f'- {label}: ditemukan {len(matches_sorted)} kandidat. Contoh: {preview_ids}')

            if chosen:
                selected_ids.append(chosen['id'])
                print(f'  -> pilih free-tier: {chosen["id"]}')
            else:
                print(f'  -> belum ada varian free-tier langsung untuk {label}, lanjut fallback.')
        else:
            print(f'- {label}: tidak ditemukan di katalog saat ini.')

    if len(selected_ids) < MAX_EVALUATOR_MODELS:
        free_pool = [m for m in catalog if _is_free_model(m)]

        # Naikkan prioritas fallback yang kita rekomendasikan.
        ordered_pool = []
        used = set(selected_ids)

        for hinted_id in FALLBACK_FREE_MODEL_HINTS:
            for m in free_pool:
                if m.get('id') == hinted_id and m.get('id') not in used:
                    ordered_pool.append(m)
                    used.add(m.get('id'))

        for m in sorted(free_pool, key=lambda x: x.get('context_length', 0), reverse=True):
            if m.get('id') not in used:
                ordered_pool.append(m)
                used.add(m.get('id'))

        for m in ordered_pool:
            if len(selected_ids) >= MAX_EVALUATOR_MODELS:
                break
            selected_ids.append(m['id'])

    # Jaga unik dan batasi ke 3 model.
    dedup = []
    for mid in selected_ids:
        if mid not in dedup:
            dedup.append(mid)
    dedup = dedup[:MAX_EVALUATOR_MODELS]

    print('\nModel evaluator terpilih:')
    for i, mid in enumerate(dedup, start=1):
        print(f'{i}. {mid}')

    return dedup


def build_eval_prompt(row):
    return f"""
Kamu adalah juri evaluator percakapan chatbot pendidikan bernama Levely.
Tugasmu adalah memberikan skor 1 hingga 5 pada balasan chatbot Levely berdasarkan pesan terakhir dari user.

Kriteria Penilaian:
- Skor 1: Balasan sama sekali tidak logis, salah kaprah, atau menolak menjawab tanpa alasan.
- Skor 2: Balasan kurang relevan dan tidak bersentuhan dengan konteks materi yang ditanyakan.
- Skor 3: Balasan cukup relevan, namun belum personal terhadap progres belajar jika dibutuhkan.
- Skor 4: Balasan bagus, relevan dengan materi, dan bahasanya ramah sesuai persona.
- Skor 5: Balasan sangat tepat, rinci, membantu, dan jika relevan mengaitkan progres/points siswa secara wajar.

Aturan tambahan:
- Nilai berdasarkan kualitas respons asisten terhadap pesan user ini (single-answer grading).
- Berikan alasan singkat maksimal 1 kalimat.
- Output WAJIB JSON valid tanpa teks tambahan.

Format output JSON ketat:
{{
  "score": <angka_1_sampai_5>,
  "reason": "<alasan singkat>"
}}

Konteks Interaksi:
- User (Nama: {row.get('user_name', 'Anonim')}, Point: {row.get('user_points', 0)}): {row.get('user_message', '')}
- Chatbot Levely: {row.get('chatbot_reply', '')}
""".strip()


def parse_score_json(raw_text):
    if not raw_text:
        raise ValueError('Empty response text')

    text = raw_text.strip()

    # Model kadang membungkus JSON dalam code fence.
    if text.startswith('```'):
        text = text.strip('`')
        text = text.replace('json\n', '', 1).strip()

    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f'Response is not JSON object: {text[:120]}')

    payload = json.loads(text[start:end + 1])
    score = payload.get('score')
    reason = payload.get('reason', '')

    score = int(score)
    if score < 1 or score > 5:
        raise ValueError(f'Score out of range: {score}')

    return score, str(reason)


def call_openrouter_judge(model_id, prompt, max_retries=2):

    payload = {
        'model': model_id,
        'messages': [
            {
                'role': 'system',
                'content': 'Anda adalah evaluator ketat yang selalu mengembalikan JSON valid sesuai format yang diminta.'
            },
            {
                'role': 'user',
                'content': prompt
            }
        ],
        'temperature': 0,
        'response_format': {'type': 'json_object'}
    }

    last_error = None
    for _ in range(1, max_retries + 1):
        try:
            resp = requests.post(OPENROUTER_API_URL, json=payload, timeout=90)
            if resp.status_code >= 400:
                last_error = RuntimeError(f'HTTP {resp.status_code}: {resp.text[:240]}')
                time.sleep(SLEEP_BETWEEN_CALLS_SEC)
                continue

            data = resp.json()
            content = data['choices'][0]['message']['content']
            score, reason = parse_score_json(content)
            time.sleep(SLEEP_BETWEEN_CALLS_SEC)
            return score, reason
        except Exception as e:
            last_error = e
            time.sleep(SLEEP_BETWEEN_CALLS_SEC)

    raise RuntimeError(f'Model {model_id} gagal setelah {max_retries} percobaan. Error terakhir: {last_error}')


def evaluate_with_models(row, model_ids):
    prompt = build_eval_prompt(row)

    scores = {}
    reasons = {}

    for model_id in model_ids:
        try:
            score, reason = call_openrouter_judge(model_id, prompt, max_retries=MAX_RETRIES_PER_MODEL)
            scores[model_id] = score
            reasons[model_id] = reason
        except Exception as e:
            reasons[model_id] = f'ERROR: {e}'

    valid_scores = [v for v in scores.values() if isinstance(v, int)]

    consensus_exact = None
    consensus_score = None
    score_std = None

    if len(valid_scores) >= MIN_VALID_JUDGES:
        score_counts = pd.Series(valid_scores).value_counts().sort_values(ascending=False)
        top_count = score_counts.iloc[0]
        top_scores = score_counts[score_counts == top_count].index.tolist()

        if len(top_scores) == 1:
            consensus_score = int(top_scores[0])
        else:
            consensus_score = int(round(pd.Series(valid_scores).mean()))

        consensus_exact = len(set(valid_scores)) == 1 and len(valid_scores) >= 2
        score_std = float(pd.Series(valid_scores).std(ddof=0)) if len(valid_scores) > 1 else 0.0

    llm_reason = ''
    if reasons:
        # Simpan ringkasan reason dari model pertama yang sukses.
        for mid in model_ids:
            if mid in scores:
                llm_reason = reasons.get(mid, '')
                break
        if not llm_reason:
            llm_reason = '; '.join([f'{k}: {v}' for k, v in reasons.items()])

    if len(valid_scores) < MIN_VALID_JUDGES:
        llm_reason = f'INSUFFICIENT_VALID_JUDGES ({len(valid_scores)}/{MIN_VALID_JUDGES}); {llm_reason}'

    return pd.Series({
        'llm_score': consensus_score,
        'llm_reason': llm_reason,
        'llm_models_used': ','.join(model_ids),
        'llm_num_models_scored': len(valid_scores),
        'llm_exact_consensus': consensus_exact,
        'llm_score_std': score_std,
        'llm_model_scores': json.dumps(scores, ensure_ascii=False),
        'llm_model_reasons': json.dumps(reasons, ensure_ascii=False)
    })


def stratified_sample(df, stratify_col, sample_size, random_state=42):
    if df.empty:
        return df.copy()

    n_target = min(sample_size, len(df))
    if stratify_col not in df.columns or n_target <= 0:
        return df.sample(n_target, random_state=random_state).copy()

    df_work = df.copy()
    df_work['_stratum_key'] = df_work[stratify_col].fillna('__MISSING__').astype(str)

    strata_sizes = df_work['_stratum_key'].value_counts(sort=False)
    proportional = strata_sizes * n_target / len(df_work)
    allocation = proportional.astype(int)

    remaining = n_target - int(allocation.sum())

    if n_target >= len(strata_sizes):
        zero_strata = allocation[allocation == 0].index.tolist()
        for key in zero_strata:
            if remaining <= 0:
                break
            allocation.loc[key] += 1
            remaining -= 1

    fractions = (proportional - proportional.astype(int)).sort_values(ascending=False)
    for key in fractions.index:
        if remaining <= 0:
            break
        allocation.loc[key] += 1
        remaining -= 1
 
    sampled_parts = []
    for key, group in df_work.groupby('_stratum_key', sort=False):
        n_take = min(int(allocation.get(key, 0)), len(group))
        if n_take > 0:
            sampled_parts.append(group.sample(n=n_take, random_state=random_state))

    if not sampled_parts:
        return df_work.sample(n_target, random_state=random_state).drop(columns=['_stratum_key']).copy()

    sampled = pd.concat(sampled_parts, ignore_index=False)

    # Backfill jika pembulatan menyebabkan jumlah sample kurang dari target.
    if len(sampled) < n_target:
        remaining_pool = df_work.drop(index=sampled.index)
        if not remaining_pool.empty:
            sampled = pd.concat(
                [sampled, remaining_pool.sample(min(n_target - len(sampled), len(remaining_pool)), random_state=random_state)],
                ignore_index=False,
            )

    sampled = sampled.head(n_target).copy()
    sampled = sampled.drop(columns=['_stratum_key'])
    return sampled


def compute_intermodel_agreement(df_scored):
    def compute_fleiss_kappa(score_maps, categories, expected_raters=3):
        usable = []
        for score_map in score_maps:
            values = []
            for value in score_map.values():
                try:
                    values.append(int(round(float(value))))
                except Exception:
                    continue
            if len(values) == expected_raters:
                usable.append(values)

        if len(usable) < 2:
            return None, 0

        n = expected_raters
        N = len(usable)

        p_i_list = []
        category_totals = {c: 0 for c in categories}

        for item_scores in usable:
            counts = {c: 0 for c in categories}
            for s in item_scores:
                if s in counts:
                    counts[s] += 1
            p_i = (sum(v * v for v in counts.values()) - n) / (n * (n - 1))
            p_i_list.append(p_i)
            for c in categories:
                category_totals[c] += counts[c]

        p_bar = sum(p_i_list) / N
        p_e = sum((category_totals[c] / (N * n)) ** 2 for c in categories)
        denom = 1 - p_e
        if denom <= 0:
            return None, N

        return (p_bar - p_e) / denom, N

    rows = []
    score_maps = []
    for idx, row in df_scored.iterrows():
        try:
            score_map = json.loads(row.get('llm_model_scores', '{}') or '{}')
            score_maps.append(score_map)
            for model_id, score in score_map.items():
                rows.append({'row_id': idx, 'model_id': model_id, 'score': float(score)})
        except Exception:
            continue

    if not rows:
        print('Belum ada data skor antar-model untuk hitung agreement.')
        return

    long_df = pd.DataFrame(rows)
    pivot = long_df.pivot_table(index='row_id', columns='model_id', values='score', aggfunc='first')

    model_cols = list(pivot.columns)
    if len(model_cols) < 2:
        print('Minimal perlu 2 model dengan skor valid untuk hitung agreement antar-model.')
        return

    print('\nAgreement antar-model evaluator:')

    pair_reports = []
    for m1, m2 in itertools.combinations(model_cols, 2):
        pair = pivot[[m1, m2]].dropna()
        if pair.empty:
            continue

        exact_match = (pair[m1].round() == pair[m2].round()).mean()
        kappa = cohen_kappa_score(pair[m1].round().astype(int), pair[m2].round().astype(int), weights='quadratic')
        pair_reports.append({
            'model_a': m1,
            'model_b': m2,
            'n_overlap': len(pair),
            'exact_match': exact_match,
            'weighted_kappa': kappa
        })

    if pair_reports:
        pair_df = pd.DataFrame(pair_reports).sort_values(['weighted_kappa', 'exact_match'], ascending=False)
        display(pair_df)

    fleiss_kappa, n_items = compute_fleiss_kappa(score_maps, categories=[1, 2, 3, 4, 5], expected_raters=min(3, len(model_cols)))
    if fleiss_kappa is not None:
        print(f'Fleiss kappa ({n_items} item, {min(3, len(model_cols))} rater): {fleiss_kappa:.3f}')
    else:
        print('Fleiss kappa belum bisa dihitung (butuh >=2 item dengan jumlah rater yang sama).')

    consensus_rate = (df_scored['llm_exact_consensus'] == True).mean() if 'llm_exact_consensus' in df_scored.columns else None
    if consensus_rate is not None:
        print(f'Exact consensus (semua model valid memberi skor sama): {consensus_rate:.2%}')


df_sample = pd.DataFrame()
selected_model_ids = []

if not df_pairs.empty:
    df_sample = stratified_sample(df_pairs, STRATIFY_COL, SAMPLE_SIZE, random_state=42)
    df_sample = df_sample.sort_values(['session_id', 'turn_index']).reset_index(drop=True)

    print(f"Stratified sampling aktif: kolom={STRATIFY_COL}, target={SAMPLE_SIZE}, terambil={len(df_sample)}")

    if RUN_LLM_EVAL:
        if not BACKEND_LLM_API_BASE:
            print('BACKEND_LLM_API_BASE belum tersedia. Isi env lalu jalankan ulang.')
        else:
            try:
                smoke = run_backend_route_smoke_test()
                if not smoke['ok']:
                    print('Smoke test endpoint backend gagal. Evaluasi dibatalkan.')
                    for d in smoke['details']:
                        print('-', d)
                    raise RuntimeError('Backend route smoke test failed.')

                catalog = fetch_openrouter_model_catalog()
                selected_model_ids = select_evaluator_models(catalog)

                if JUDGE_MODE != 'single' and len(selected_model_ids) < 3:
                    print('Peringatan: model free-tier yang tersedia kurang dari 3. Evaluasi tetap dilanjutkan.')

                if len(selected_model_ids) < MIN_VALID_JUDGES:
                    print(f'Jumlah evaluator valid kurang dari MIN_VALID_JUDGES={MIN_VALID_JUDGES}. Evaluasi dibatalkan.')
                    selected_model_ids = []

                if not selected_model_ids:
                    print('Tidak ada model evaluator yang bisa dipilih dari katalog OpenRouter.')
                else:
                    df_eval_run = df_sample.head(min(len(df_sample), MAX_ITEMS_PER_RUN)).copy()
                    stop_reason = 'completed'
                    expected_calls = len(df_eval_run) * len(selected_model_ids)
                    if expected_calls > MAX_TOTAL_JUDGE_CALLS:
                        max_rows = max(1, MAX_TOTAL_JUDGE_CALLS // max(1, len(selected_model_ids)))
                        df_eval_run = df_eval_run.head(max_rows).copy()

                    print(f'Memulai penilaian otomatis untuk {len(df_eval_run)} sampel dengan model: {selected_model_ids}')

                    eval_rows = []
                    item_failures = 0
                    consecutive_failures = 0
                    for _, eval_row in df_eval_run.iterrows():
                        eval_series = evaluate_with_models(eval_row, selected_model_ids)
                        n_valid = int(eval_series.get('llm_num_models_scored', 0) or 0)

                        if n_valid < MIN_VALID_JUDGES:
                            item_failures += 1
                            consecutive_failures += 1
                        else:
                            consecutive_failures = 0

                        eval_rows.append(eval_series)

                        if consecutive_failures >= MAX_CONSECUTIVE_ITEM_FAILURES:
                            print(f'Hard stop: consecutive failures mencapai {MAX_CONSECUTIVE_ITEM_FAILURES}.')
                            stop_reason = 'hard_stop_consecutive_failures'
                            break

                    eval_cols = pd.DataFrame(eval_rows)
                    df_eval_run = df_eval_run.head(len(eval_cols)).reset_index(drop=True)
                    df_sample = pd.concat([df_eval_run, eval_cols.reset_index(drop=True)], axis=1)

                    if len(df_sample) > 0:
                        failure_rate = item_failures / len(df_sample)
                        if failure_rate > MAX_ITEM_FAILURE_RATE:
                            print(f'Peringatan: failure rate tinggi ({failure_rate:.2%}) di atas batas {MAX_ITEM_FAILURE_RATE:.0%}.')
                            stop_reason = 'high_failure_rate_warning'

                    successful_items = int((pd.to_numeric(df_sample.get('llm_num_models_scored', 0), errors='coerce').fillna(0) >= MIN_VALID_JUDGES).sum()) if len(df_sample) > 0 else 0
                    failed_items = int(len(df_sample) - successful_items)
                    run_summary = pd.DataFrame([
                        {
                            'run_timestamp_utc': pd.Timestamp.utcnow().isoformat(),
                            'validation_mode': VALIDATION_MODE,
                            'judge_mode': JUDGE_MODE,
                            'backend_llm_api_base': BACKEND_LLM_API_BASE,
                            'selected_models': len(selected_model_ids),
                            'requested_rows': int(min(len(df_pairs), SAMPLE_SIZE)),
                            'evaluated_rows': int(len(df_sample)),
                            'estimated_calls': int(len(df_sample) * len(selected_model_ids)),
                            'successful_items': successful_items,
                            'failed_items': failed_items,
                            'failure_rate': float((failed_items / len(df_sample)) if len(df_sample) > 0 else 0.0),
                            'stop_reason': stop_reason
                        }
                    ])
                    print('Ringkasan run evaluator:')
                    display(run_summary)
                    print('Evaluasi multi-model selesai.')
                    display(df_sample.head())
            except Exception as e:
                print('Gagal menjalankan evaluasi multi-model:', e)
    else:
        print('RUN_LLM_EVAL=False, melewati pemanggilan evaluator model.')

# Ringkasan agregasi jika skor sudah ada
if not df_sample.empty and 'llm_score' in df_sample.columns:
    scored = df_sample.dropna(subset=['llm_score']).copy()
    if not scored.empty:
        scored['llm_score'] = scored['llm_score'].astype(float)

        print(f"Jumlah respons yang berhasil dinilai: {len(scored)}")
        print(f"Rata-rata skor konsensus evaluator: {scored['llm_score'].mean():.2f}")

        if 'llm_num_models_scored' in scored.columns:
            avg_models = pd.to_numeric(scored['llm_num_models_scored'], errors='coerce').mean()
            print(f'Rata-rata jumlah model yang sukses menilai per sampel: {avg_models:.2f}')

        user_agg = (
            scored.groupby('user_id', dropna=False)['llm_score']
            .agg(
                turns='count',
                mean_score='mean',
                median_score='median',
                std_score='std',
                pct_low=lambda s: (s <= 2).mean() * 100,
                pct_high=lambda s: (s >= 4).mean() * 100
            )
            .reset_index()
            .sort_values(['turns', 'mean_score'], ascending=[False, False])
        )

        print('\nTop user by jumlah turn yang dinilai:')
        display(user_agg.head(10))

        compute_intermodel_agreement(scored)

        plt.figure(figsize=(6, 4))
        sns.histplot(scored['llm_score'], bins=5, kde=False)
        plt.title('Distribusi Skor Konsensus LLM (1-5)')
        plt.xlabel('Skor')
        plt.ylabel('Jumlah Respons')
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 4))
        sns.boxplot(data=scored, x='llm_score')
        plt.title('Sebaran Skor Konsensus LLM')
        plt.xlabel('Skor')
        plt.tight_layout()
        plt.show()
    else:
        print('Belum ada skor valid dari evaluator model.')

Stratified sampling aktif: kolom=user_id, target=30, terambil=14
RUN_LLM_EVAL=False, melewati pemanggilan evaluator model.


## 5. Ekspor untuk Human Evaluation (Opsional, jika consistency_human)
Bagian ini hanya dijalankan jika VALIDATION_MODE='consistency_human'.

In [ ]:
if VALIDATION_MODE != 'consistency_human':
    print("VALIDATION_MODE='consistency_only' -> lewati ekspor human evaluation.")
elif not df_sample.empty:
    export_cols = [
        'eval_id',
        'session_id',
        'turn_index',
        'user_id',
        'user_name',
        'user_points',
        'user_message',
        'chatbot_reply',
        'created_at'
    ]
    existing_export_cols = [c for c in export_cols if c in df_sample.columns]

    df_export = df_sample[existing_export_cols].copy()
    df_export['human_score'] = ''

    try:
        df_export.to_excel('human_evaluation_sample.xlsx', index=False)
        print('Berhasil diekspor ke human_evaluation_sample.xlsx')
        print('Silakan isi kolom human_score (1-5) secara manual.')
    except Exception as e:
        print('Failed to export:', e)
else:
    print('df_sample kosong, tidak ada data untuk diekspor.')

Berhasil diekspor ke human_evaluation_sample.xlsx
Silakan isi kolom human_score (1-5) secara manual.


## 6. Hitung Validasi Human vs LLM (Opsional, jika consistency_human)
Jalankan hanya saat VALIDATION_MODE='consistency_human' dan file human score sudah terisi.

In [ ]:
from pathlib import Path

human_file = Path('human_evaluation_sample.xlsx')

if VALIDATION_MODE != 'consistency_human':
    print("VALIDATION_MODE='consistency_only' -> lewati validasi human vs LLM.")
elif df_sample.empty:
    print('df_sample kosong. Jalankan tahap sampling/evaluasi terlebih dahulu.')
elif 'llm_score' not in df_sample.columns:
    print('Kolom llm_score belum ada. Aktifkan RUN_LLM_EVAL=True lalu jalankan evaluasi LLM.')
elif not human_file.exists():
    print('File human_evaluation_sample.xlsx belum ditemukan. Jalankan tahap ekspor terlebih dahulu.')
else:
    try:
        df_human = pd.read_excel(human_file)

        required_human_cols = {'eval_id', 'human_score'}
        if not required_human_cols.issubset(df_human.columns):
            raise ValueError(f'Kolom wajib pada file human eval: {required_human_cols}')

        df_eval = pd.merge(
            df_sample,
            df_human[['eval_id', 'human_score']],
            on='eval_id',
            how='inner',
            suffixes=('', '_humanfile')
        ).copy()

        # Normalisasi skor
        df_eval['llm_score'] = pd.to_numeric(df_eval['llm_score'], errors='coerce')
        df_eval['human_score'] = pd.to_numeric(df_eval['human_score'], errors='coerce')
        df_eval = df_eval.dropna(subset=['llm_score', 'human_score'])

        if df_eval.empty:
            print('Belum ada pasangan skor llm_score dan human_score yang valid.')
        else:
            # Pastikan rentang score 1-5
            df_eval = df_eval[(df_eval['llm_score'].between(1, 5)) & (df_eval['human_score'].between(1, 5))]

            if df_eval.empty:
                print('Skor valid 1-5 tidak ditemukan setelah filtering.')
            else:
                pearson_corr = df_eval['llm_score'].corr(df_eval['human_score'])
                exact_match = (df_eval['llm_score'].round() == df_eval['human_score'].round()).mean()
                kappa = cohen_kappa_score(
                    df_eval['llm_score'].round().astype(int),
                    df_eval['human_score'].round().astype(int),
                    weights='quadratic'
                )
                avg_llm_score = df_eval['llm_score'].mean()
                avg_human_score = df_eval['human_score'].mean()

                print(f'Jumlah data tervalidasi: {len(df_eval)}')
                print(f'Pearson Correlation (Human vs LLM): {pearson_corr:.3f}')
                print(f'Quadratic Weighted Cohen Kappa: {kappa:.3f}')
                print(f'Exact Match: {exact_match:.2%}')
                print(f'Rata-rata skor LLM: {avg_llm_score:.2f} dari 5.0')
                print(f'Rata-rata skor Human: {avg_human_score:.2f} dari 5.0')

                # Simpan hasil gabungan untuk audit
                df_eval.to_csv('llm_vs_human_comparison.csv', index=False)
                print('File tersimpan: llm_vs_human_comparison.csv')

                # Visualisasi distribusi dan korelasi
                fig, axes = plt.subplots(1, 2, figsize=(12, 4))

                sns.countplot(data=df_eval, x='llm_score', palette='Blues', ax=axes[0])
                axes[0].set_title('Distribusi Skor LLM')
                axes[0].set_xlabel('Skor LLM')
                axes[0].set_ylabel('Frekuensi')

                sns.scatterplot(data=df_eval, x='human_score', y='llm_score', alpha=0.7, ax=axes[1])
                axes[1].set_title('Human vs LLM Score')
                axes[1].set_xlabel('Human Score')
                axes[1].set_ylabel('LLM Score')

                plt.tight_layout()
                plt.show()

    except Exception as e:
        print('Gagal memproses validasi human vs LLM:', e)

Kolom llm_score belum ada. Aktifkan RUN_LLM_EVAL=True lalu jalankan evaluasi LLM.


In [11]:
# One-off cleanup: flush Supabase chat tables used by this notebook.
def delete_all_rows_by_id(client, table_name, id_col='id', batch_size=500):
    total_deleted = 0
    while True:
        rows = client.table(table_name).select(id_col).range(0, batch_size - 1).execute().data or []
        ids = [r.get(id_col) for r in rows if r.get(id_col) is not None]
        if not ids:
            break
        client.table(table_name).delete().in_(id_col, ids).execute()
        total_deleted += len(ids)
    return total_deleted

if supabase is None:
    print('Supabase client is not initialized. Cannot flush tables.')
else:
    try:
        # Delete children first to satisfy FK constraints.
        deleted_messages = delete_all_rows_by_id(supabase, 'chat_messages', id_col='id')
        deleted_sessions = delete_all_rows_by_id(supabase, 'chat_sessions', id_col='id')

        remaining_messages = fetch_all_rows(supabase, 'chat_messages')
        remaining_sessions = fetch_all_rows(supabase, 'chat_sessions')
        print(f'Deleted chat_messages: {deleted_messages}')
        print(f'Deleted chat_sessions: {deleted_sessions}')
        print(f'Remaining chat_messages: {len(remaining_messages)}')
        print(f'Remaining chat_sessions: {len(remaining_sessions)}')
    except Exception as e:
        print('Flush error:', e)

Deleted chat_messages: 28
Deleted chat_sessions: 15
Remaining chat_messages: 0
Remaining chat_sessions: 0


In [16]:
# Seed mock chat history for all users across a full week of usage.
import random
import uuid
from datetime import datetime, timedelta, timezone

RESET_TABLES_BEFORE_SEED = True
SIM_DAYS = 7
MIN_SESSIONS_PER_USER_PER_DAY = 0
MAX_SESSIONS_PER_USER_PER_DAY = 2
MIN_TURNS_PER_SESSION = 1   # 1 turn = 2 messages (user + assistant)
MAX_TURNS_PER_SESSION = 4
RANDOM_SEED = 42

def _flush_chat_tables(client):
    def delete_all_rows_by_id(table_name, id_col='id', batch_size=500):
        total_deleted = 0
        while True:
            rows = client.table(table_name).select(id_col).range(0, batch_size - 1).execute().data or []
            ids = [r.get(id_col) for r in rows if r.get(id_col) is not None]
            if not ids:
                break
            client.table(table_name).delete().in_(id_col, ids).execute()
            total_deleted += len(ids)
        return total_deleted

    deleted_messages = delete_all_rows_by_id('chat_messages')
    deleted_sessions = delete_all_rows_by_id('chat_sessions')
    return deleted_sessions, deleted_messages

def seed_mock_week_history(
    client,
    users_df,
    sim_days=7,
    min_sessions_per_day=0,
    max_sessions_per_day=2,
    min_turns_per_session=1,
    max_turns_per_session=4,
    seed=42,
):
    if client is None:
        raise RuntimeError('Supabase client is not initialized.')
    if users_df is None or users_df.empty:
        raise RuntimeError('df_users kosong. Jalankan cell MySQL terlebih dahulu.')
    if sim_days < 1:
        raise ValueError('sim_days minimal 1.')
    if min_sessions_per_day < 0 or max_sessions_per_day < min_sessions_per_day:
        raise ValueError('Range session per day tidak valid.')
    if min_turns_per_session < 1 or max_turns_per_session < min_turns_per_session:
        raise ValueError('Range turn per session tidak valid.')

    rng = random.Random(seed)
    now = datetime.now(timezone.utc)
    base_date = (now - timedelta(days=sim_days - 1)).replace(hour=0, minute=0, second=0, microsecond=0)

    user_rows = users_df[['id', 'name', 'points']].copy()
    user_rows['name'] = user_rows['name'].fillna('Anonim')
    user_rows['points'] = pd.to_numeric(user_rows['points'], errors='coerce').fillna(0)

    prompt_templates = [
        'Halo Levely, materi apa yang sedang saya pelajari?',
        'Boleh jelaskan ringkas topik ini?',
        'Contoh soal sederhana untuk topik ini apa ya?',
        'Kenapa konsep ini penting untuk dipahami?',
        'Bisa kasih tips belajar cepat untuk bagian ini?',
        'Saya masih bingung di bagian inti, bisa diulang?',
        'Kalau mau latihan, mulai dari mana dulu?',
        'Apa kesalahan umum yang harus saya hindari?',
    ]

    sessions_payload = []
    messages_payload = []
    sessions_per_user = []
    turns_per_session = []
    daily_stats = []

    for _, u in user_rows.iterrows():
        user_id = u['id']
        user_name = str(u['name'])
        user_points = float(u['points'])
        user_session_count = 0

        for day_idx in range(sim_days):
            day_start = base_date + timedelta(days=day_idx)
            session_count_today = rng.randint(min_sessions_per_day, max_sessions_per_day)
            daily_stats.append({'date': day_start.date().isoformat(), 'user_id': str(user_id), 'sessions': session_count_today})

            for _ in range(session_count_today):
                session_id = str(uuid.uuid4())
                user_session_count += 1
                n_turns = rng.randint(min_turns_per_session, max_turns_per_session)
                turns_per_session.append(n_turns)

                sessions_payload.append({
                    'id': session_id,
                    'user_id': user_id
                })

                # Random session start time inside this day.
                start_offset_minutes = rng.randint(0, 23 * 60 + 59)
                session_start = day_start + timedelta(minutes=start_offset_minutes)

                for t in range(n_turns):
                    prompt = rng.choice(prompt_templates)
                    reply = (
                        f'Halo {user_name}, untuk pertanyaanmu: {prompt} '
                        f'Saran Levely: pahami konsep inti dulu, lihat satu contoh, lalu kerjakan 1 latihan kecil. '
                        f'Poinmu saat ini {user_points:.0f}.'
                    )
                    msg_time = (session_start + timedelta(minutes=t * rng.randint(2, 8))).isoformat()

                    messages_payload.append({
                        'id': str(uuid.uuid4()),
                        'session_id': session_id,
                        'role': 'user',
                        'content': prompt,
                        'metadata': {'mock': True, 'turn': t + 1, 'sim_day': day_idx + 1},
                        'token_count': None,
                        'created_at': msg_time
                    })
                    messages_payload.append({
                        'id': str(uuid.uuid4()),
                        'session_id': session_id,
                        'role': 'assistant',
                        'content': reply,
                        'metadata': {'mock': True, 'turn': t + 1, 'sim_day': day_idx + 1},
                        'token_count': None,
                        'created_at': msg_time
                    })

        sessions_per_user.append(user_session_count)

    if sessions_payload:
        client.table('chat_sessions').insert(sessions_payload).execute()
    if messages_payload:
        client.table('chat_messages').insert(messages_payload).execute()

    return {
        'created_sessions': len(sessions_payload),
        'created_messages': len(messages_payload),
        'sim_days': sim_days,
        'min_sessions_per_user': min(sessions_per_user) if sessions_per_user else 0,
        'max_sessions_per_user': max(sessions_per_user) if sessions_per_user else 0,
        'avg_sessions_per_user': (sum(sessions_per_user) / len(sessions_per_user)) if sessions_per_user else 0,
        'min_turns_per_session': min(turns_per_session) if turns_per_session else 0,
        'max_turns_per_session': max(turns_per_session) if turns_per_session else 0,
        'avg_turns_per_session': (sum(turns_per_session) / len(turns_per_session)) if turns_per_session else 0,
        'daily_stats': daily_stats,
    }

try:
    if RESET_TABLES_BEFORE_SEED:
        deleted_sessions, deleted_messages = _flush_chat_tables(supabase)
        print(f'Reset done -> deleted chat_sessions: {deleted_sessions}, chat_messages: {deleted_messages}')

    seed_result = seed_mock_week_history(
        supabase,
        df_users,
        sim_days=SIM_DAYS,
        min_sessions_per_day=MIN_SESSIONS_PER_USER_PER_DAY,
        max_sessions_per_day=MAX_SESSIONS_PER_USER_PER_DAY,
        min_turns_per_session=MIN_TURNS_PER_SESSION,
        max_turns_per_session=MAX_TURNS_PER_SESSION,
        seed=RANDOM_SEED,
    )

    daily_df = pd.DataFrame(seed_result['daily_stats'])
    daily_summary = daily_df.groupby('date', as_index=False)['sessions'].sum().sort_values('date')

    print(f"Simulated days: {seed_result['sim_days']}")
    print(f"Created chat_sessions: {seed_result['created_sessions']}")
    print(f"Created chat_messages: {seed_result['created_messages']}")
    print(
        'Sessions per user -> '
        f"min: {seed_result['min_sessions_per_user']}, "
        f"max: {seed_result['max_sessions_per_user']}, "
        f"avg: {seed_result['avg_sessions_per_user']:.2f}"
    )
    print(
        'Turns per session -> '
        f"min: {seed_result['min_turns_per_session']}, "
        f"max: {seed_result['max_turns_per_session']}, "
        f"avg: {seed_result['avg_turns_per_session']:.2f}"
    )
    print('Daily sessions summary (7 days):')
    display(daily_summary)
except Exception as e:
    print('Mock seed error:', e)

Reset done -> deleted chat_sessions: 56, chat_messages: 328
Simulated days: 7
Created chat_sessions: 391
Created chat_messages: 1988
Sessions per user -> min: 3, max: 12, avg: 6.98
Turns per session -> min: 1, max: 4, avg: 2.54
Daily sessions summary (7 days):


,date,sessions
0,2026-03-05,51
1,2026-03-06,60
2,2026-03-07,57
3,2026-03-08,58
4,2026-03-09,54
5,2026-03-10,47
6,2026-03-11,64


## 7. Proxy Credibility Check (Tanpa Human Label)
Bagian ini memberi indikator *sementara* untuk contextuality dan adaptivity ketika human score belum tersedia. Ini **bukan pengganti validasi human**, tapi berguna sebagai bukti internal awal.

In [ ]:
import re


def _safe_text(v):
    if pd.isna(v):
        return ''
    return str(v)


def compute_proxy_credibility(df_pairs_input, df_sample_input=None):
    if df_pairs_input is None or df_pairs_input.empty:
        raise ValueError('df_pairs kosong. Jalankan tahap pairing terlebih dahulu.')

    work = df_pairs_input.copy()
    for col in ['user_message', 'chatbot_reply', 'user_name', 'user_points', 'created_at', 'user_id', 'session_id']:
        if col not in work.columns:
            work[col] = '' if col != 'user_points' else 0

    # Contextuality proxy: lexical overlap sederhana antara prompt user dan jawaban asisten.
    token_pattern = re.compile(r"\b\w+\b", re.UNICODE)

    def lexical_overlap_ratio(user_msg, bot_msg):
        user_tokens = {t.lower() for t in token_pattern.findall(_safe_text(user_msg)) if len(t) > 2}
        bot_tokens = {t.lower() for t in token_pattern.findall(_safe_text(bot_msg)) if len(t) > 2}
        if not user_tokens:
            return 0.0
        overlap = user_tokens.intersection(bot_tokens)
        return len(overlap) / len(user_tokens)

    work['context_overlap'] = work.apply(
        lambda r: lexical_overlap_ratio(r.get('user_message', ''), r.get('chatbot_reply', '')),
        axis=1
    )

    # Personalization proxy: apakah jawaban menyebut nama user atau poin user.
    def has_personalization(row):
        reply = _safe_text(row.get('chatbot_reply', '')).lower()
        name = _safe_text(row.get('user_name', '')).strip().lower()
        points_str = str(int(float(row.get('user_points', 0)))) if str(row.get('user_points', '')).strip() != '' else ''

        name_hit = bool(name) and name != 'anonim' and name in reply
        points_hit = bool(points_str) and points_str in reply and ('poin' in reply or 'point' in reply)
        return int(name_hit or points_hit)

    work['personalization_hit'] = work.apply(has_personalization, axis=1)

    # Basic quality proxy: panjang jawaban yang terlalu pendek cenderung kurang membantu.
    work['reply_len'] = work['chatbot_reply'].astype(str).str.len()
    work['adequate_length'] = (work['reply_len'] >= 60).astype(int)

    # Adaptive trend proxy: apakah quality proxy naik seiring turn index per user.
    if 'turn_index' in work.columns:
        work['turn_index_num'] = pd.to_numeric(work['turn_index'], errors='coerce')
    else:
        work['turn_index_num'] = 1

    per_user_corr = []
    for _, g in work.groupby('user_id', dropna=False):
        g2 = g.dropna(subset=['turn_index_num'])
        if len(g2) < 3:
            continue
        score_proxy = 0.5 * g2['context_overlap'] + 0.3 * g2['personalization_hit'] + 0.2 * g2['adequate_length']
        c = g2['turn_index_num'].corr(score_proxy)
        if pd.notna(c):
            per_user_corr.append(float(c))

    adaptivity_corr_avg = float(pd.Series(per_user_corr).mean()) if per_user_corr else None

    # Jika llm_score tersedia, tambahkan confidence proxy berbasis konsensus.
    llm_confidence = None
    if df_sample_input is not None and not df_sample_input.empty and 'llm_score' in df_sample_input.columns:
        scored = df_sample_input.dropna(subset=['llm_score']).copy()
        if not scored.empty:
            exact_consensus_rate = None
            std_inverse = None

            if 'llm_exact_consensus' in scored.columns:
                exact_consensus_rate = float((scored['llm_exact_consensus'] == True).mean())
            if 'llm_score_std' in scored.columns:
                score_std = pd.to_numeric(scored['llm_score_std'], errors='coerce').dropna()
                if not score_std.empty:
                    # Makin kecil std antar-model, makin tinggi confidence.
                    std_inverse = float((1 / (1 + score_std)).mean())

            if exact_consensus_rate is not None or std_inverse is not None:
                parts = [p for p in [exact_consensus_rate, std_inverse] if p is not None]
                llm_confidence = float(sum(parts) / len(parts))

    summary = {
        'n_pairs': int(len(work)),
        'contextuality_overlap_mean': float(work['context_overlap'].mean()),
        'contextuality_overlap_p50': float(work['context_overlap'].median()),
        'personalization_rate': float(work['personalization_hit'].mean()),
        'adequate_length_rate': float(work['adequate_length'].mean()),
        'adaptivity_trend_corr_mean': adaptivity_corr_avg,
        'llm_confidence_proxy': llm_confidence,
    }

    return work, summary


try:
    proxy_df, proxy_summary = compute_proxy_credibility(df_pairs, df_sample)

    print('Proxy credibility summary (tanpa human label):')
    print(f"- Jumlah pair: {proxy_summary['n_pairs']}")
    print(f"- Context overlap mean: {proxy_summary['contextuality_overlap_mean']:.3f}")
    print(f"- Context overlap median: {proxy_summary['contextuality_overlap_p50']:.3f}")
    print(f"- Personalization rate: {proxy_summary['personalization_rate']:.2%}")
    print(f"- Adequate reply length rate: {proxy_summary['adequate_length_rate']:.2%}")

    if proxy_summary['adaptivity_trend_corr_mean'] is None:
        print('- Adaptivity trend (corr): belum cukup data per user (min 3 turn/user).')
    else:
        print(f"- Adaptivity trend (corr rata-rata): {proxy_summary['adaptivity_trend_corr_mean']:.3f}")

    if proxy_summary['llm_confidence_proxy'] is None:
        print('- LLM confidence proxy: belum tersedia (jalankan RUN_LLM_EVAL=True).')
    else:
        print(f"- LLM confidence proxy: {proxy_summary['llm_confidence_proxy']:.3f}")

    print('\nContoh baris proxy:')
    display(proxy_df[['eval_id', 'user_id', 'turn_index', 'context_overlap', 'personalization_hit', 'adequate_length']].head(10))
except Exception as e:
    print('Proxy credibility check error:', e)

Proxy credibility summary (tanpa human label):
- Jumlah pair: 14
- Context overlap mean: 0.602
- Context overlap median: 0.667
- Personalization rate: 28.57%
- Adequate reply length rate: 100.00%
- Adaptivity trend (corr rata-rata): 0.278
- LLM confidence proxy: belum tersedia (jalankan RUN_LLM_EVAL=True).

Contoh baris proxy:


,eval_id,user_id,turn_index,context_overlap,personalization_hit,adequate_length
0,16b45838-7640-414b-a22d-52f3f92a8da4_1,4.0,1,0.500000,0,1
1,16b45838-7640-414b-a22d-52f3f92a8da4_2,4.0,2,0.666667,0,1
2,1e64d54e-1d18-401d-9d74-5fbf29a723e1_1,4.0,1,1.000000,0,1
3,76faea2a-e314-451d-9cf6-e5aa73c29896_1,4.0,1,0.600000,0,1
4,76faea2a-e314-451d-9cf6-e5aa73c29896_2,4.0,2,1.000000,0,1
5,933de30d-5771-4a23-a7bf-d879bc3ac25c_1,4.0,1,1.000000,1,1
6,cd7d7b80-3125-4ce7-ab71-518ea4f1fa1d_1,4.0,1,0.666667,1,1
7,dbaed5de-264e-43a1-b5e9-3a4b928c1e46_1,4.0,1,0.000000,1,1
8,dbaed5de-264e-43a1-b5e9-3a4b928c1e46_2,4.0,2,1.000000,1,1
9,e851dba0-58e3-430e-abb6-1d0508b96089_1,4.0,1,0.000000,0,1
